In [ ]:
import os
from dotenv import load_dotenv
import re
from typing import Any

from langchain_openai import OpenAI, ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import (
    PIIMiddleware, 
    HumanInTheLoopMiddleware, 
    AgentState, 
    hook_config, 
    AgentMiddleware
)
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langgraph.runtime import Runtime

In [3]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Safety mechanisms that control what goes into, and out of an AI Agent or LLM
1. Two main methods
    * Deterministic - rule based algorithms (regex, keyword matching)
    * LLM Based - symantic meaning
- process only safe, appropriate inputs
- only perform approved actions
- only returns validated, compliant outputs

They are implemented as **middleware** that intercepts execution:
- **Before** the agent starts (input guardrails)
- **After** it completes (output guardrails)
- **Around** model and tool calls

### Common Use Cases:
| Use Case | Example |
|---|---|
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |

In [5]:
# Deterministic
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked"""

    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of Thailand?",
    "Explain how malware spreads.",
]

print("--- Deterministic Guardrail Test ---")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "❌ BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

--- Deterministic Guardrail Test ---
❌ BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of Thailand?
❌ BLOCKED: Explain how malware spreads.


In [14]:
# Model based
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety.  Returns SAFE or UNSAFE"""
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
    prompt = f"""Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'.

    Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("--- Model Based Guardrail Test ---")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "❌ UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")


--- Model Based Guardrail Test ---
❌ UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of Thailand?
✅ SAFE: Explain how malware spreads.


# LangChain Built in Guardrails

### Supported PII Types:
| Type | Example |
|---|---|
| `email` | user@example.com |
| `credit_card` | 5105-1051-0510-5100 |
| `ip` | 192.168.1.1 |
| `mac_address` | 00:1A:2B:3C:4D:5E |
| `url` | https://secret-site.com |

### Strategies:
| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |


In [6]:
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information"""
    return f"Customer record located for query: {query}"


# Agent with PII Middleware
agent = create_agent(
    model="gpt-5-mini",
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to the model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise exception if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [ ]:
# PII redaction

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card number is 5105-1051-0510-5100.  Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I can help — but first I need a bit more detail. What do you want help with related to that email and card (billing question, suspected fraud, update payment method, cancel subscription, something else)?

A few important notes before we proceed:
- Don’t post full card numbers or other sensitive data here. The last 4 digits (****-****-****-5100) are fine for identification.
- If this is an urgent fraud/compromise issue, contact your bank or card issuer immediately to block the card.

If you want immediate steps, here are common actions depending on the problem:

If you suspect fraud or your card was used without authorization
- Contact your card issuer or bank right away and ask them to freeze or cancel the card and open a fraud claim.
- Review recent transactions and note any unauthorized charges.
- If charges are unauthorized, file a dispute with the bank and ask about getting a replacement card.
- Change passwords and enable two-factor authentication (2FA) on a

In [22]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card number is ****-****-****-5100.  Can you help me?', additional_kwargs={}, response_metadata={}, id='7171ea4d-8404-4999-8569-5b13fe347bef'),
  AIMessage(content='I can help — but first I need a bit more detail. What do you want help with related to that email and card (billing question, suspected fraud, update payment method, cancel subscription, something else)?\n\nA few important notes before we proceed:\n- Don’t post full card numbers or other sensitive data here. The last 4 digits (****-****-****-5100) are fine for identification.\n- If this is an urgent fraud/compromise issue, contact your bank or card issuer immediately to block the card.\n\nIf you want immediate steps, here are common actions depending on the problem:\n\nIf you suspect fraud or your card was used without authorization\n- Contact your card issuer or bank right away and ask them to freeze or cancel the card and open a fraud claim.\n- Review

In [7]:
# API key blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"❌ Blocked API Key in plain text as expected: {e}")

❌ Blocked API Key in plain text as expected: Detected 1 instance(s) of api_key in text content


---
## 👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

**Best for:**
- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

**Key requirement:** A `checkpointer` for state persistence across interrupts.

In [8]:
@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send email to a recipient"""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database"""
    return f"Deleted records from {table} where {condition}"


# Create agent with HITL middleware
hitl_agent = create_agent(
    model="gpt-5-mini",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                # True = HITL approval needed, False = auto-approve
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        ),
    ],
    checkpointer = InMemorySaver(), # required for state persistence
)

print("Human-in-the-loop agent created!")

Human-in-the-loop agent created!


In [9]:
# Invoke - the agent will pause before sending for approval
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results."}]},
    config=config
)

print("=== Agent Response Paused - awaiting human approval ===")
print(result)

=== Agent Response Paused - awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results.', additional_kwargs={}, response_metadata={}, id='875a50df-87bd-48e4-bde1-7a121372749f'), AIMessage(content='I can send that — before I do, a couple quick questions so the email is right:\n\n1. Do you want me to send a generic summary (placeholders for numbers) or include the actual Q4 figures and details? If the latter, please provide the numbers / a report or tell me where to pull it from.\n2. Who should the sender appear to be? (Your name / default / specific address)\n3. Any CC or BCC recipients?\n4. Any attachments to include?\n5. Tone: formal, neutral, or celebratory?\n6. Do you want a call-to-action (e.g., meeting invite, review request, approve budget)?\n\nBelow is a suggested draft you can approve or edit. Tell me when you want me to send it as-is or provide the details to fill in.\n\nSuggested subject:\nQ4 Results — Summary and Ne

In [32]:
# Human review and approval
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config # Same thread ID
)


print("=== Approved! Final Response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final Response ===
I can send that. Do you want me to send a default message or customize it first? Here’s a suggested draft — tell me any changes, or confirm and I’ll send:

Subject: Q4 Results — Summary and Next Steps

Hi team,

Please find the Q4 results attached for your review. Key items to note:
- Please review the results and flag any questions or discrepancies.
- Share any feedback by [feedback deadline].
- We will discuss the results and action items in our Q4 review meeting on [proposed date/time]. If that time doesn’t work for you, let me know.

Thanks,
[Your name]

Questions for you:
- Confirm sender name (what should appear in the From line)?
- Feedback deadline (date/time)?
- Meeting date/time (or “TBD”)?
- Attach a file? If yes, provide filename or paste content to attach.
- Any CC or BCC addresses?
- Send now or save draft?

Reply with the details or just say “send as-is” and I’ll send the default message.


In [10]:
# Human rejection
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final Response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final Response ===
I won’t run a destructive operation without explicit confirmation.

Before I proceed, please confirm:
- Do you want me to actually delete the records now, or do you want the SQL/commands so you can run them yourself?
- If you want me to run them, reply with: "Yes — delete the records now." (this explicit confirmation is required).
- Also tell me the database type (PostgreSQL, MySQL, SQLite, etc.) so I use the correct syntax/format.

If you just want the SQL or a safe preview, here are safe options you can run:

- Preview how many rows would be deleted:
  - SQL: SELECT COUNT(*) FROM users WHERE active = false;
  - Note: In MySQL if active is TINYINT use active = 0 instead.

- Backup before deleting (simple copy):
  - PostgreSQL: CREATE TABLE users_backup AS TABLE users;
  - MySQL: CREATE TABLE users_backup AS SELECT * FROM users;

- Delete (inside a transaction so you can rollback if needed):
  - BEGIN;
    DELETE FROM users WHERE active = false;
    -- 

---
## ⚙️ Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)

Use `before_agent()` to validate or block requests **before any LLM processing begins**.

**Best for:**
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [11]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs before the agent processes anything - zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower()for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None: 
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"❌ Blocked - keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content."
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

In [12]:
@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Results for: {query}"

# Create agent with content filter
filtered_agent = create_agent(
    model="gpt-5-mini",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created")

Content filter agent created


In [13]:
# test 1 - Safe request should process successfully
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response")
print(result["messages"][-1].content)

✅ Safe request response
Short answer
Machine learning (ML) is a field of computer science that gives computers the ability to learn patterns and make predictions or decisions from data, without being explicitly programmed for each specific task.

How it works (in plain terms)
- You collect data that represents the problem (examples with inputs and sometimes outputs).
- You choose a model (an algorithm that can represent relationships in the data).
- You train the model on the data so it learns a mapping from inputs to outputs or finds structure in the inputs.
- You evaluate the model on new data to see how well it generalizes.
- You use the trained model to make predictions or take actions.

Main types of ML
- Supervised learning: model learns from labeled examples (input → known output). Used for classification (spam vs. not spam) and regression (predict house price).
- Unsupervised learning: model finds structure in unlabeled data (clustering, dimensionality reduction). Example: cust

In [14]:
# test 2 - unsafe request that should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})

print("❌ Unsafe request response:")
print(result["messages"][-1].content)

❌ Blocked - keyword detected: 'hack'
❌ Unsafe request response:
I cannot process requests containing inappropriate content.Please rephrase your request.


---
## 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)

Use `after_agent()` to validate the final agent response **before the user sees it**.

**Best for:**
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [15]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs after the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        self.safety_model = ChatOpenAI(model="gpt-5-mini", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.  Respond
        only with 'SAFE' or 'UNSAFE'.
        
        Response to evaluate:
        {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None

In [16]:
@tool
def general_tool(query: str) -> str:
    """A general purpose tool"""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model="gpt-5-mini",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware(),]
)

print("Output safety agent created")

Output safety agent created


In [17]:
# test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather like today in Los Angeles?"}]
})

print("Response:")
print(result["messages"][-1].content)

Response:
I can’t fetch live weather from here. If you want the current, up-to-the-minute conditions I can either (a) tell you how to check quickly yourself, or (b) give a typical/expected description for Los Angeles at this time of year.

Quick options to get live data now
- Google: search “weather Los Angeles” or “weather [ZIP]”  
- National Weather Service: https://www.weather.gov/ (enter Los Angeles or ZIP)  
- Weather sites/apps: Weather.com, AccuWeather, Dark Sky replacements, or your phone’s Weather app  
- Terminal: run curl "wttr.in/Los+Angeles" (or wttr.in/Los+Angeles?format=3 for a one-line summary)

Typical weather for Los Angeles in late August (what to expect today)
- General: Warm to hot, mostly sunny, low chance of rain.  
- Coastal neighborhoods (Santa Monica, Venice): highs ~68–76°F (20–24°C), often a cool marine layer/fog in the morning that clears by midday.  
- Inland (Downtown LA, Burbank, San Fernando Valley): highs ~85–95°F (29–35°C), sometimes higher during hea

---
## 🧱 Section 7: Layered / Combined Guardrails

Stack multiple guardrails in the `middleware=[]` array. They execute **in order**, building layered protection.

```
User Input
    ↓
[Layer 1] ContentFilterMiddleware    ← Deterministic input filter
    ↓
[Layer 2] PIIMiddleware (input)      ← PII redaction on input
    ↓
[Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
    ↓
[Layer 4] PIIMiddleware (output)     ← PII redaction on output
    ↓
[Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
    ↓
User Response
```

In [19]:
@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Seach results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email"""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with 5-layer guardrails created")

Agent with 5-layer guardrails created
